In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [38]:
import os
import json
import shutil
import traceback
import pandas as pd
import xml.etree.ElementTree as ET
from collections import defaultdict
from music21 import converter, midi

In [39]:
# Load metadata JSON
json_path = '/content/drive/MyDrive/slovenian-folk-song-ballads/slovenian-folk-song-ballads.json'
with open(json_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)['pieces']

# Group by piece number and first lyrics
grouped_by_number = defaultdict(list)
grouped_by_lyrics = defaultdict(list)

for pid, data in metadata.items():
    num = data['opus'].get('piece:num', 'Unknown')
    first_line = data['opus'].get('variant:first', '').strip()
    grouped_by_number[num].append(pid)
    if first_line:
        grouped_by_lyrics[first_line].append(pid)

# Create DataFrame of duplicates by piece number
duplicates_by_number = {k: v for k, v in grouped_by_number.items() if len(v) > 1}
df_duplicates_num = pd.DataFrame([
    {'piece:num': k, 'variants': ", ".join(v)} for k, v in duplicates_by_number.items()
])

In [40]:
mei_root = '/content/drive/MyDrive/slovenian-folk-song-ballads'
with_dup_root = os.path.join(mei_root, 'grouped_by_region_with_duplicates')
no_dup_root = os.path.join(mei_root, 'grouped_by_region_without_duplicates')
os.makedirs(with_dup_root, exist_ok=True)
os.makedirs(no_dup_root, exist_ok=True)

# Organize by region and piece number
group_by_region = defaultdict(list)
for pid, data in metadata.items():
    region = data['opus'].get('origin:region', 'UnknownRegion').replace(" ", "_")
    piece_num = data['opus'].get('piece:num', 'Unknown')
    score_path = data['sources'][0]['score'].split('/')[-1]
    group_by_region[region].append({
        'filename': score_path,
        'piece_id': pid,
        'region': region,
        'piece_num': piece_num
    })

# Map MEI filenames to full paths
mei_files = {}
for root, dirs, files in os.walk(mei_root):
    for file in files:
        if file.endswith('.mei'):
            mei_files[file] = os.path.join(root, file)
for region, songs in group_by_region.items():
    region_dir = os.path.join(with_dup_root, region)
    os.makedirs(region_dir, exist_ok=True)
    for song in songs:
        src = mei_files.get(song['filename'])
        dst = os.path.join(region_dir, song['filename'])
        if src:
            shutil.copy2(src, dst)
for region, songs in group_by_region.items():
    seen_piece_nums = set()
    region_dir = os.path.join(no_dup_root, region)
    os.makedirs(region_dir, exist_ok=True)
    for song in sorted(songs, key=lambda x: x['piece_id']):
        if song['piece_num'] not in seen_piece_nums:
            seen_piece_nums.add(song['piece_num'])
            src = mei_files.get(song['filename'])
            dst = os.path.join(region_dir, song['filename'])
            if src:
                shutil.copy2(src, dst)

In [42]:
def remove_lyrics_from_mei(mei_data):
    try:
        root = ET.fromstring(mei_data)
        ns = {'mei': 'http://www.music-encoding.org/ns/mei'}

        for syl in root.findall('.//mei:syl', ns):
            wordpos = syl.get('wordpos')
            if wordpos not in ['s', 'i', 'm', 't']:
                syl.set('wordpos', 'i' if wordpos == 'u' else 's')

        for verse in root.findall('.//mei:verse', ns):
            for ancestor in root.iter():
                if verse in list(ancestor):
                    ancestor.remove(verse)
                    break

        for measure in root.findall('.//mei:measure', ns):
            measure_num = measure.get('n')
            if measure_num:
                try:
                    int(measure_num)
                except ValueError:
                    parent = None
                    for ancestor in root.iter():
                        if measure in list(ancestor):
                            parent = ancestor
                            break
                    if parent is not None:
                        siblings = parent.findall('mei:measure', ns)
                        index = siblings.index(measure)
                        measure.set('n', str(index))

        return ET.tostring(root, encoding='utf-8').decode('utf-8')
    except Exception as e:
        print(f"Error processing MEI: {e}")
        traceback.print_exc()
        return mei_data

def convert_mei_to_midi(input_path, output_path=None):
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            mei_data = f.read()

        cleaned_mei = remove_lyrics_from_mei(mei_data)
        temp_file = input_path + '.temp.mei'
        with open(temp_file, 'w', encoding='utf-8') as f:
            f.write(cleaned_mei)

        score = converter.parse(temp_file, format='mei')

        if output_path is None:
            output_path = os.path.splitext(input_path)[0] + '.mid'

        mf = midi.translate.music21ObjectToMidiFile(score)
        mf.open(output_path, 'wb')
        mf.write()
        mf.close()

        os.remove(temp_file)
        return output_path
    except Exception as e:
        print(f"Failed to convert {input_path}: {e}")
        traceback.print_exc()
        return None

In [43]:
def batch_convert_meis(input_dir, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    success_count = 0
    failed_count = 0

    for root, dirs, files in os.walk(input_dir):
        for filename in files:
            if filename.lower().endswith('.mei'):
                input_path = os.path.join(root, filename)
                rel_path = os.path.relpath(root, input_dir)
                output_subdir = os.path.join(output_dir, rel_path)
                os.makedirs(output_subdir, exist_ok=True)
                output_filename = os.path.splitext(filename)[0] + '.mid'
                output_path = os.path.join(output_subdir, output_filename)

                result = convert_mei_to_midi(input_path, output_path)
                if result:
                    success_count += 1
                else:
                    failed_count += 1

    print(f"Converted: {success_count} succeeded, {failed_count} failed.")
    return success_count

In [44]:
input_mei_without_duplicates = '/content/drive/MyDrive/slovenian-folk-song-ballads/grouped_by_region_without_duplicates'
output_midi_without_duplicates = '/content/drive/MyDrive/slovenian-folk-song-ballads/midi_without_duplicates'

batch_convert_meis(input_mei_without_duplicates, output_midi_without_duplicates)

Converted: 107 succeeded, 0 failed.


107

In [ ]:
input_mei_with_duplicates = '/content/drive/MyDrive/slovenian-folk-song-ballads/grouped_by_region_with_duplicates'
output_midi_with_duplicates = '/content/drive/MyDrive/slovenian-folk-song-ballads/midi_with_duplicates'

batch_convert_meis(input_mei_with_duplicates, output_midi_with_duplicates)